In [59]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check
import sys
from pathlib import Path

# Subo un nivel desde /etl para que se vea /common
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from common.loaders import load_valid_country_ids


API_URL = "https://restcountries.com/v3.1/all?fields=cca3,altSpellings"



def validate_altspellings(df: pd.DataFrame) -> pd.DataFrame:

    valid_ids = load_valid_country_ids()
    print(df)
    schema = pa.DataFrameSchema(
        {
            "id_country": Column(
                str,
                checks=Check.isin(valid_ids), nullable=False
            ),
            "alt_spelling": Column(
                str,
                checks=Check.str_length(min_value=1),  # no vacío
                nullable=False,
            ),
        }
    )

    
    return schema.validate(df)


@task
def extract_altspellings():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_altspellings(data):
    # Crear DataFrame
    df = pd.DataFrame(data)

    # Explode: 1 fila por altSpelling
    df_alt = df.explode("altSpellings")

    # Renombrar columnas
    df_alt = df_alt.rename(columns={
        "cca3": "id_country",
        "altSpellings": "alt_spelling"
    })

    return df_alt

@task
def load_altspellings(df: pd.DataFrame):
    file_path = Path("../stage/country_altspellings.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")

@flow(name="etl_altspellings_flow")
def etl_altspellings():
    data = extract_altspellings()
    df = transform_altspellings(data)
    df = validate_altspellings(df)
    load_altspellings(df)

if __name__ == "__main__":
    etl_altspellings()


03:21:28.955 | INFO    | Flow run 'nostalgic-serval' - Beginning flow run 'nostalgic-serval' for flow 'etl_altspellings_flow'

03:21:30.079 | INFO    | Task run 'extract_altspellings-282' - Finished in state Completed()

03:21:30.312 | INFO    | Task run 'transform_altspellings-deb' - Finished in state Completed()

    id_country          alt_spelling
0          JAM                    JM
1          COM                    KM
1          COM  Union of the Comoros
1          COM     Union des Comores
1          COM      Udzima wa Komori
..         ...                   ...
248        SGP             Singapura
248        SGP    Republik Singapura
248        SGP                新加坡共和国
249        LBR                    LR
249        LBR   Republic of Liberia

[789 rows x 2 columns]
Saved 789 rows to ..\stage\country_altspellings.csv


03:21:30.540 | INFO    | Task run 'load_altspellings-bbc' - Finished in state Completed()

03:21:30.554 | INFO    | Flow run 'nostalgic-serval' - Finished in state Completed()